# Review Collection Log

This notebook reviews the output of `02_data_collection.ipynb` to identify data quality issues — failed pulls, missing fields, and insufficient history — and decide how each affected company should be handled (ticker correction, exclusion, or acceptance with a noted limitation).

In [30]:
import sys
sys.path.append("..")

from src import config

import pandas as pd

## Identify Failed Pulls

Filter the collection log for any entries with `status == "fail"` and list the affected tickers. Each one is investigated individually below — most failures are expected to be due to incorrect, delisted, or renamed tickers.

In [3]:
collection_log = pd.read_csv("../data/raw/collection_log.csv")
failures = collection_log[collection_log["status"] == "fail"]
print(failures["ticker"].unique())

<StringArray>
[]
Length: 0, dtype: str


### Removed Companies — Delisted/Acquired

The following companies failed data collection due to delisting following acquisition or take-private transactions, confirmed via manual lookup. Removed from the universe as they are no longer investable.

- **DNB** (Dun & Bradstreet) — acquired by Clearlake Capital, August 2025
- **SXS.L** (Spectris) — acquired by KKR, December 2025
- **TET.L** (Treatt) — acquired by Döhler, delisted July 2026
- **RWI.L** (Renewi) — acquired by Macquarie/BCI consortium, June 2025
- **APH.L** (Alliance Pharma) — acquired by DBAY Advisors, May 2025
- **ALPH.L** (Alpha Group) — acquired by Corpay, October 2025

### Replaced

- **MEG → ONT** — Montrose Environmental renamed to Onterris Inc.
- **FI → FISV** — Fiserv's correct yfinance ticker is FISV, not FI. Original entry was a data-entry error, not a delisting.

In [ ]:
partial_fields = collection_log[
    (collection_log["status"] == "success") & 
    (collection_log["missing_required_fields"].notna()) &
    (collection_log["missing_required_fields"] != "[]")
]
print(partial_fields[["ticker", "missing_required_fields"]])

     ticker                            missing_required_fields
16    RMV.L                 ['Current Debt', 'Long Term Debt']
18    RMV.L                                   ['Gross Profit']
76    SCT.L                 ['Current Debt', 'Long Term Debt']
86   BYIT.L                 ['Current Debt', 'Long Term Debt']
162   DGE.L                       ['Stock Based Compensation']
167  BATS.L                       ['Stock Based Compensation']
186  FEVR.L                                 ['Long Term Debt']
191   GRG.L                                   ['Current Debt']
196  NICL.L                                 ['Long Term Debt']
356     WTS                                   ['Current Debt']
366     SXI                                   ['Current Debt']
373  ITRK.L                                   ['Gross Profit']
376   RCP.L                 ['Current Debt', 'Long Term Debt']
377   RCP.L  ['Depreciation And Amortization', 'Stock Based...
378   RCP.L  ['Gross Profit', 'Operating Income', 'EBIT

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
fixed_partial = partial_fields[partial_fields["ticker"] == "RCP.L"]
print(fixed_partial[["ticker", "missing_required_fields"]])
pd.reset_option("display.max_colwidth")
pd.reset_option("display.max_rows")

    ticker                                        missing_required_fields
376  RCP.L                             ['Current Debt', 'Long Term Debt']
377  RCP.L  ['Depreciation And Amortization', 'Stock Based Compensation']
378  RCP.L         ['Gross Profit', 'Operating Income', 'EBIT', 'EBITDA']


## Review: Missing Required Fields

Companies flagged with `status == "success"` but at least one missing required field, across balance sheet, cash flow, and income statement pulls. Most cases involve a single missing field (commonly "Current Debt," "Long Term Debt," or "Stock Based Compensation" — fields some companies don't report separately). A few companies (e.g. RCP.L) are missing several fields across multiple statements and warrant closer review — likely a smaller/less-covered company with genuinely limited data.

**Decision:** single missing fields are accepted as a known limitation (handled downstream by treating missing data as unavailable for that factor, rather than excluding the company). Companies missing several core fields across multiple statements are reviewed individually.

In [29]:
fundamentals_log = collection_log[collection_log["data_type"].isin(["balance_sheet", "cash_flow", "income_statement"])]
fundamentals_summary = fundamentals_log.groupby("ticker").agg(
    min_rows = ("rows", "min"),
    start_date = ("start_date", "min"),
    end_date = ("end_date", "max")
)
print(fundamentals_summary.sort_values("min_rows"))
print(fundamentals_summary.sort_values("start_date"))
print(fundamentals_summary.sort_values("end_date"))

        min_rows           start_date             end_date
ticker                                                    
RSW.L        3.0  2022-06-30 00:00:00  2024-06-30 00:00:00
SCT.L        3.0  2022-07-31 00:00:00  2024-07-31 00:00:00
NCC.L        3.0  2022-05-31 00:00:00  2025-09-30 00:00:00
ABF.L        4.0  2023-08-31 00:00:00  2025-08-31 00:00:00
AIT          4.0  2022-06-30 00:00:00  2025-06-30 00:00:00
...          ...                  ...                  ...
WCN          4.0  2022-12-31 00:00:00  2025-12-31 00:00:00
WDAY         4.0  2022-01-31 00:00:00  2026-01-31 00:00:00
WM           4.0  2022-12-31 00:00:00  2025-12-31 00:00:00
WST          4.0  2021-12-31 00:00:00  2025-12-31 00:00:00
WTS          4.0  2022-12-31 00:00:00  2025-12-31 00:00:00

[121 rows x 3 columns]
        min_rows           start_date             end_date
ticker                                                    
SGE.L        4.0  2021-09-30 00:00:00  2025-09-30 00:00:00
TTEK         4.0  2021-09-30 00:

In [ ]:
num_fields = {
    "balance_sheet": len(config.BALANCE_SHEET_REQUIRED_FIELDS),
    "cash_flow": len(config.CASH_FLOW_REQUIRED_FIELDS),
    "income_statement": len(config.INCOME_STATEMENT_REQUIRED_FIELDS)
}

partial_values = collection_log[collection_log["missing_values"] > 0].sort_values("missing_values", ascending = False)
# partial_values["total_cells"] = partial_values["rows"] * 
print(partial_values[["ticker", "data_type", "status", "missing_values"]])

     ticker         data_type   status  missing_values
96   TRST.L     balance_sheet  success            13.0
13   AUTO.L  income_statement  success            12.0
123    ADSK  income_statement  success            12.0
546   MER.L     balance_sheet  success            11.0
188  FEVR.L  income_statement  success            10.0
116     NOW     balance_sheet  success            10.0
126    WDAY     balance_sheet  success            10.0
563     ADP  income_statement  success            10.0
461     CRL     balance_sheet  success            10.0
258   ROR.L  income_statement  success            10.0
8    EXPN.L  income_statement  success            10.0
468     WST  income_statement  success            10.0
426  CTEC.L     balance_sheet  success             9.0
256   ROR.L     balance_sheet  success             9.0
411    TTEK     balance_sheet  success             9.0
93    NCC.L  income_statement  success             9.0
136    SNPS     balance_sheet  success             9.0
121    ADS